# 01 — Baseline generation (Task 2)

Accelerator: **GPU T4 x2**, Internet **on**. Generates responses from the base model
`Qwen/Qwen2.5-0.5B` on the fixed evaluation sets with the shared decoding settings in
`configs/generation.yaml` (greedy). Outputs go to `outputs/generations/`.

When it finishes: **Save Version** so the outputs persist, and keep the zip.

In [ ]:
REPO_URL = "https://github.com/madgod2004/small-model-alignment-starter.git"
REPO_DIR = "/kaggle/working/small-model-alignment-starter"
BRANCH = "main"

In [ ]:
import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib "peft>=0.11.0" "trl>=0.12.0" pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
!python scripts/check_env.py --expect_gpus 2

## Sanity check (2–5 prompts)
Inspect these by hand: prompt format, EOS/stop behaviour, length.

In [ ]:
!python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --generator qwen2.5-0.5b-base     --eval_set eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/sanity_base.json     --limit 3 --print --overwrite

## Full baseline generation
AlpacaEval (300, seed 42) on `cuda:0` and the HH eval set on `cuda:1`, in parallel. Rerunning resumes from checkpoints / skips finished files.

In [ ]:
%%bash
mkdir -p outputs/generations
python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --generator qwen2.5-0.5b-base     --eval_set eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/base_alpaca_eval.json     --device cuda:0 > outputs/gen_base_alpaca.log 2>&1 &
python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --generator qwen2.5-0.5b-base     --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/generations/base_hh_eval.json     --device cuda:1 > outputs/gen_base_hh.log 2>&1 &
wait
tail -n 2 outputs/gen_base_alpaca.log outputs/gen_base_hh.log

In [ ]:
import json
for name in ["base_alpaca_eval", "base_hh_eval"]:
    rows = json.load(open(f"outputs/generations/{name}.json"))
    meta = json.load(open(f"outputs/generations/{name}.meta.json"))
    print(name, len(rows), "entries |", meta["num_truncated"], "hit max_new_tokens |", meta["generation_seconds"], "s")
    print("  sample:", rows[0]["instruction"][:80], "->", rows[0]["output"][:200].replace("
", " "))

In [ ]:
!rm -f outputs/generations/sanity_base*.json
!cd outputs && zip -qr /kaggle/working/baseline_generations.zip generations *.log && ls -lh /kaggle/working/*.zip